In [1]:
%reload_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np, pretty_midi
from pianotrans.pipeline import transcribe
from pianotrans.rhythm import estimate_tempo, quantise

tempo_ok, onset_ok, dur_ok = [], [], []
for wav in sorted(Path("../data/test").glob("*.wav")):
    pm = pretty_midi.PrettyMIDI(str(wav.with_suffix(".mid")))
    true_bpm = pm.get_tempo_changes()[1][0]
    ref = pm.instruments[0].notes

    notes = transcribe(str(wav))
    bpm = estimate_tempo([n.onset for n in notes])
    tempo_ok.append(abs(bpm - true_bpm) / true_bpm < 0.04)
    if len(notes) != len(ref):
        continue                       # can't compare note-by-note if counts differ

    q = quantise(notes, bpm)
    t0 = ref[0].start
    true_on = np.round([(n.start - t0) * true_bpm / 60 * 4 for n in ref]) / 4
    true_dur = np.append(np.diff(true_on), np.round((ref[-1].end - ref[-1].start) / 0.9 * true_bpm / 60 * 4) / 4)
    onset_ok.append(np.mean([n.onset for n in q] == true_on))
    dur_ok.append(np.mean([n.offset - n.onset for n in q] == true_dur))

print(f"tempo within 4%: {np.mean(tempo_ok):.3f}  onsets on correct beat: {np.mean(onset_ok):.3f}  "
      f"lengths correct: {np.mean(dur_ok):.3f}  ({len(onset_ok)} files compared)")

tempo within 4%: 1.000  onsets on correct beat: 1.000  lengths correct: 0.991  (27 files compared)
